# Zero-Inflated and Two-Part Mixed Effects Models (Python)

**Author:** Dimitris Rizopoulos

This notebook accompanies the R vignette *Zero-Inflated and Two-Part Mixed
Effects Models* and demonstrates the Python API in **glmmadaptive**.

`ZIPoisson` and `ZINegativeBinomial` are **fully implemented**.  The hurdle
families are stubs that will be added in a future release.

| Family (R) | Python class | Status |
|---|---|---|
| `zi.poisson()` | `ZIPoisson` | **Implemented** |
| `zi.negative.binomial()` | `ZINegativeBinomial` | **Implemented** |
| `zi.binomial()` | `ZIBinomial` | Stub — not yet implemented |
| `hurdle.lognormal()` | `HurdleLogNormal` | Stub — not yet implemented |
| `hurdle.poisson()` | `HurdlePoisson` | Stub — not yet implemented |
| `hurdle.negative.binomial()` | `HurdleNegativeBinomial` | Stub — not yet implemented |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import nbinom

# Simulate longitudinal count data with structural zeros
rng = np.random.default_rng(42)
n, K = 100, 8
ids  = np.repeat(np.arange(n), K)
sex  = np.repeat(rng.integers(0, 2, n), K).astype(float)
time = np.tile(np.arange(K), n).astype(float)

b = rng.normal(0, np.sqrt(0.5), n)
eta_y = 1.0 + 0.3 * sex + 0.15 * time + b[ids]
mu_y  = np.exp(eta_y)

# NB counts (shape = 3)
shape = 3.0
p_nb  = shape / (shape + mu_y)
y = nbinom.rvs(n=shape, p=p_nb, random_state=rng).astype(float)

# Structural zeros
pi = expit(-1.0 + 0.5 * sex)
zi_mask = rng.binomial(1, pi).astype(bool)
y[zi_mask] = 0.0

DF = pd.DataFrame({"id": ids, "sex": sex, "time": time, "y": y})
print(DF.head())
print(f"\nZero proportion: {(DF.y == 0).mean():.2f}")

---

## 1  Zero-Inflated Poisson Mixed Effects Model

The ZIP model uses a logistic regression to model structural zeros and a
Poisson distribution for the non-zero counts.  Pass `family=ZIPoisson()` and
`zi_fixed` (and optionally `zi_random`) to `MixedModel`.

In [ ]:
from glmmadaptive import MixedModel
from glmmadaptive.families import ZIPoisson
from glmmadaptive.results import MixModResults

# fm1: ZIP with sex in the zero part (no ZI random effect)
fm1 = MixedModel(
    fixed    = "y ~ sex * time",
    random   = "~ 1 | id",
    data     = DF,
    family   = ZIPoisson(),
    zi_fixed = "~ sex",
    control  = {"iter_em": 30, "verbose": False},
).fit()

print(fm1.summary())

In [ ]:
# fm2: extend with a random intercept in the zero part
fm2 = MixedModel(
    fixed     = "y ~ sex * time",
    random    = "~ 1 | id",
    data      = DF,
    family    = ZIPoisson(),
    zi_fixed  = "~ sex",
    zi_random = "~ 1 | id",
    control   = {"iter_em": 30, "verbose": False},
).fit()

# Likelihood ratio test
print(MixModResults.anova(fm1, fm2))

---

## 2  Zero-Inflated Negative Binomial Mixed Effects Model

The ZINB model extends ZIP by adding an over-dispersion parameter
$\theta = \exp(\phi)$ to the non-zero part.  Substitute
`ZINegativeBinomial()` for `ZIPoisson()`:

In [ ]:
from glmmadaptive.families import ZINegativeBinomial

# gm1: ZINB with sex in the zero part
gm1 = MixedModel(
    fixed    = "y ~ sex * time",
    random   = "~ 1 | id",
    data     = DF,
    family   = ZINegativeBinomial(),
    zi_fixed = "~ sex",
    control  = {"iter_em": 30, "verbose": False},
).fit()

print(gm1.summary())
print(f"\nEstimated theta = exp(phis[0]) = {np.exp(gm1.phis[0]):.3f}")

In [ ]:
# Non-nested comparison: ZINB (gm1) vs ZIP+ZI random effect (fm2)
print(MixModResults.anova(gm1, fm2))

---

## 3  Two-Part Mixed Effects Model for Semi-Continuous Data (HurdleLogNormal)

Models continuous data with excess zeros: logistic regression for the
zero/non-zero split and a log-normal mixed model for the positive part.
The dispersion parameter `exp(phis)` gives the standard deviation of the
log-normal errors.

> **Note:** `HurdleLogNormal` is not yet implemented in the Python port.

**R reference:**

```r
km1 <- mixed_model(
    y ~ sex * time, random = ~ 1 | id, data = DF,
    family = hurdle.lognormal(), n_phis = 1, zi_fixed = ~ sex
)
km2 <- update(km1, random = ~ 1 || id, zi_random = ~ 1 | id)
marginal_coefs(km2)
```

**Planned Python API:**

```python
# Raises NotImplementedError in current version
from glmmadaptive.families import HurdleLogNormal

km1 = MixedModel(
    fixed    = "y ~ sex * time",
    random   = "~ 1 | id",
    data     = DF,
    family   = HurdleLogNormal(),
    zi_fixed = "~ sex",
).fit()
```

---

## 4  Two-Part / Hurdle Poisson Mixed Effects Model

Uses a logistic regression for the zero/non-zero split and a zero-truncated
Poisson for the positive counts.  Fixed-effects coefficients relate to the
mean $\mu$ of the **full** (untruncated) Poisson, not to the conditional
mean $\mu / (1 - e^{-\mu})$ among positive counts.

> **Note:** `HurdlePoisson` is not yet implemented in the Python port.

**R reference:**

```r
dm1 <- mixed_model(
    y ~ sex * time, random = ~ time | id, data = DF,
    family = hurdle.poisson(), zi_fixed = ~ sex
)
dm2 <- update(dm1, zi_random = ~ 1 | id)
anova(dm1, dm2)
```

**Planned Python API:**

```python
# Raises NotImplementedError in current version
from glmmadaptive.families import HurdlePoisson

dm1 = MixedModel(
    fixed    = "y ~ sex * time",
    random   = "~ time | id",
    data     = DF,
    family   = HurdlePoisson(),
    zi_fixed = "~ sex",
).fit()
```

---

## 5  Two-Part / Hurdle Negative Binomial Mixed Effects Model

Identical in structure to the hurdle Poisson family, but replaces the
zero-truncated Poisson with a zero-truncated negative binomial distribution
to accommodate over-dispersion in the positive counts.

> **Note:** `HurdleNegativeBinomial` is not yet implemented in the Python port.

**R reference:**

```r
hm1 <- mixed_model(
    y ~ sex * time, random = ~ time | id, data = DF,
    family = hurdle.negative.binomial(), zi_fixed = ~ sex
)
hm2 <- update(hm1, zi_random = ~ 1 | id)
anova(hm1, hm2)
```

**Planned Python API:**

```python
# Raises NotImplementedError in current version
from glmmadaptive.families import HurdleNegativeBinomial

hm1 = MixedModel(
    fixed    = "y ~ sex * time",
    random   = "~ time | id",
    data     = DF,
    family   = HurdleNegativeBinomial(),
    zi_fixed = "~ sex",
).fit()
```